# Path 1: Customer signup & account lifecycle

Covers `POST/GET/GET :id/PUT/PATCH/DELETE /customers`.

`customers` is the single account/signup entity (email + password) since the old
standalone `users` table was merged into it. Uses Faker to simulate several
customers signing up in bulk, then exercises the full lifecycle on one of them.

Run top-to-bottom (e.g. `jupyter nbconvert --to notebook --execute 01_customer_signup_flow.ipynb`).

In [1]:
import os
import random

import requests
from faker import Faker

BASE_URL = os.environ.get("API_BASE_URL", "http://localhost:3002/api")
fake = Faker()
SIGNUP_COUNT = int(os.environ.get("SIGNUP_COUNT", "100000"))


def assert_no_password_leak(payload):
    if isinstance(payload, list):
        for item in payload:
            assert_no_password_leak(item)
        return
    assert "password" not in payload, f"password leaked in response: {payload}"
    assert "password_hash" not in payload, f"password_hash leaked in response: {payload}"


print(f"BASE_URL={BASE_URL} SIGNUP_COUNT={SIGNUP_COUNT}")

BASE_URL=http://api-service:5000/api SIGNUP_COUNT=100000


## Step 1 - Bulk sign-up: several customers register with Faker emails/passwords

In [2]:
signed_up_customers = []

for _ in range(SIGNUP_COUNT):
    payload = {
        "email": fake.unique.email(),
        "password": fake.password(length=14),
    }
    resp = requests.post(f"{BASE_URL}/customers", json=payload)
    assert resp.status_code == 201, f"signup failed: {resp.status_code} {resp.text}"
    customer = resp.json()
    assert_no_password_leak(customer)
    signed_up_customers.append(customer)

print(f"Signed up {len(signed_up_customers)} customers")
signed_up_customers[:3]

AssertionError: signup failed: 500 {"error":"Internal server error"}

## Step 2 - List all customers and confirm the new sign-ups are present

In [ ]:
resp = requests.get(f"{BASE_URL}/customers")
assert resp.status_code == 200, resp.text
all_customers = resp.json()
assert_no_password_leak(all_customers)

all_ids = {c["id"] for c in all_customers}
signed_up_ids = {c["id"] for c in signed_up_customers}
assert signed_up_ids.issubset(all_ids), "not all signed-up customers appear in the list"
print(f"{len(all_customers)} total customers, {len(signed_up_ids)} newly signed up")

## Step 3 - Fetch a single sign-up by id

In [ ]:
target = random.choice(signed_up_customers)
resp = requests.get(f"{BASE_URL}/customers/{target['id']}")
assert resp.status_code == 200, resp.text
fetched = resp.json()
assert_no_password_leak(fetched)
assert fetched["email"] == target["email"]
print("Fetched customer:", fetched)

## Step 4 - Full update (PUT): rotate email + password

In [ ]:
new_email = fake.unique.email()
new_password = fake.password(length=16)
resp = requests.put(
    f"{BASE_URL}/customers/{target['id']}",
    json={"email": new_email, "password": new_password},
)
assert resp.status_code == 200, resp.text
updated = resp.json()
assert_no_password_leak(updated)
assert updated["email"] == new_email
target = updated
print("Updated (PUT) customer:", updated)

## Step 5 - Partial update (PATCH): change email only

In [ ]:
patched_email = fake.unique.email()
resp = requests.patch(f"{BASE_URL}/customers/{target['id']}", json={"email": patched_email})
assert resp.status_code == 200, resp.text
patched = resp.json()
assert_no_password_leak(patched)
assert patched["email"] == patched_email
target = patched
print("Patched customer:", patched)

## Step 6 - Check this customer's orders (expected empty, no orders placed yet)

In [ ]:
resp = requests.get(f"{BASE_URL}/customers/{target['id']}/orders")
assert resp.status_code == 200, resp.text
orders = resp.json()
assert orders == [], f"expected no orders yet, got: {orders}"
print("Orders for this customer:", orders)

## Step 7 - Delete the customer and verify removal

In [ ]:
resp = requests.delete(f"{BASE_URL}/customers/{target['id']}")
assert resp.status_code == 200, resp.text
deleted = resp.json()
assert_no_password_leak(deleted)
print("Deleted customer:", deleted)

resp = requests.get(f"{BASE_URL}/customers/{target['id']}")
assert resp.status_code == 404, f"expected 404 after delete, got {resp.status_code}"
print("Confirmed 404 after delete")

remaining_signed_up = [c for c in signed_up_customers if c["id"] != target["id"]]
print(f"{len(remaining_signed_up)} customers from this run remain (left for other notebooks/manual cleanup).")